<a href="https://colab.research.google.com/github/sheashea16/AlphaCapture/blob/main/CaptureZero.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## AlphaCapture



*   AlphaCapture uses a minimax algorith with alpha-beta pruning to find the "best" move or sequence given the state of the mancala board
*   The depth of how many moves deep it can go can be selected, the higher the depth the better performance the agent shows.



In [1]:
# clone my git repo
!git clone https://github.com/sheashea16/AlphaCapture.git

fatal: destination path 'AlphaCapture' already exists and is not an empty directory.


In [3]:
import sys
sys.path.insert(0, "/content/AlphaCapture")

from alphacapture import AlphaCapture

In [49]:
alpha = AlphaCapture(max_player=0, depth=3)
state = (
    [4, 4, 4, 4, 4, 4, 0,
     4, 4, 4, 4, 4, 4, 0],
    0  # player 0's turn
)
seq = alpha.best_sequence(state)
print("Best move sequence:", seq)

Best move sequence: [2, 5]


In [53]:
from google.colab import ai

prompt = f"""
You are my skilled Mancala friend.

Your job:
- Explain why the chosen move sequence is the best option for the current player in this scenario.
- Do NOT invent moves (only explain the given sequence).
- Explain captures, extra turns, and positional advantages.
- Speak like a calm, experienced coach teaching a student.
- Give really short, concise, non-technical responses.
- Maximum of 2-3 sentences.

Game rules:
- Mancala with capture rules

Current board state (indices 0–5 are my pits, 6 is my store):
{state}

Chosen move sequence:
{seq}

"""

response = ai.generate_text(prompt, model_name='google/gemini-2.5-flash-lite')
print(response)

That's a smart play. You grab four stones from pit 2, which is a good capture, and then land on your store for an extra turn. This sets you up nicely for the next round.


# CaptureZero



*   CaptureZero uses a deep-Q network and was trained against AlphaCapture on varying depths in order to become "better" than it.
*   It had no previous rules of the game, other than what moves were legal, it had no knowldege of what "capturing" means and wasn't rewarded for it.



In [7]:
import sys
sys.path.insert(0, "/content/CaptureZero")

from capturezero import CaptureZero

In [54]:
zero = CaptureZero(model_path="/content/AlphaCapture/dqn_mancala.pt", dqn_player=0)
state = (
    [2, 5, 1, 7, 3, 6, 14,
     4, 0, 8, 2, 5, 1, 11],
    0
)
seq = zero.best_sequence(state)
print("Best move sequence:", seq)

Best move sequence: [4]


In [55]:
from google.colab import ai

prompt = f"""
You are my skilled Mancala friend.

Your job:
- Explain why the chosen move sequence is the best option for the current player in this scenario.
- Do NOT invent moves (only explain the given sequence).
- Explain captures, extra turns, and positional advantages.
- Speak like a calm, experienced coach teaching a student.
- Give really short, concise, non-technical responses.
- Maximum of 2-3 sentences.

Game rules:
- Mancala with capture rules

Current board state (indices 0–5 are my pits, 6 is my store):
{state}

Chosen move sequence:
{seq}

"""

response = ai.generate_text(prompt, model_name='google/gemini-2.5-flash-lite')
print(response)

That's a good start. Moving from pit 4 lets you capture your opponent's pieces. It also gives you another turn to potentially set up more captures or make a big score.


# AlphaCapture VS CaptureZero



*   Now that we've seen both models are capable of playing a "great" move given a position of the mancala board, which one is better?
*   To test this, two mock games of mancala are played between AlphaCapture and CaptureZero, each getting a turn to go first to ensure fairness.



In [59]:
# define the game logic
from alphacapture import AlphaCapture
from capturezero import CaptureZero

def print_board(board):
    print("P1 |", board[12:6:-1])
    print("   |", board[13], " " * 15, board[6])
    print("P0 |", board[0:6])
    print("-" * 40)

def play_match(
    start_player=0,
    alpha_depth=8,
    capturezero_model="/content/AlphaCapture/dqn_mancala.pt",
):
    # Initialize agents
    alpha = AlphaCapture(max_player=0, depth=alpha_depth)
    cz = CaptureZero(model_path=capturezero_model, dqn_player=1)

    # Initial state
    board = [4, 4, 4, 4, 4, 4, 0,
             4, 4, 4, 4, 4, 4, 0]
    state = (board, start_player)

    turn = 0
    print("\n=== GAME START ===\n")
    print_board(board)

    while not alpha.terminal(state):
        board, player = state
        print(f"Turn {turn} | Player {player}")

        if player == 0:
            action = alpha.best_action(state)
            agent = "AlphaCapture"
        else:
            action = cz.best_action(state)
            agent = "CaptureZero"

        if action is None:
            print("No legal moves. Ending game.")
            break

        print(f"{agent} plays pit {action}")
        state = alpha.result(state, action)
        print_board(state[0])

        turn += 1

    # Final scoring
    final_board, _ = state
    print("\n=== GAME OVER ===")
    print_board(final_board)
    print(f"Final Score → P0 (AlphaCapture): {final_board[6]} | "
          f"P1 (CaptureZero): {final_board[13]}")

    if final_board[6] > final_board[13]:
        print("AlphaCapture wins")
    elif final_board[6] < final_board[13]:
        print("CaptureZero wins")
    else:
        print("Draw")


In [77]:
# play the first match (AlphaCapture goes first)
play_match(
    start_player=0,
    alpha_depth=1,
    capturezero_model="/content/AlphaCapture/dqn_mancala.pt"
)



=== GAME START ===

P1 | [4, 4, 4, 4, 4, 4]
   | 0                 0
P0 | [4, 4, 4, 4, 4, 4]
----------------------------------------
Turn 0 | Player 0
AlphaCapture plays pit 4
P1 | [4, 4, 4, 4, 5, 5]
   | 0                 1
P0 | [4, 4, 4, 4, 0, 5]
----------------------------------------
Turn 1 | Player 1
CaptureZero plays pit 9
P1 | [5, 5, 5, 0, 5, 5]
   | 1                 1
P0 | [4, 4, 4, 4, 0, 5]
----------------------------------------
Turn 2 | Player 1
CaptureZero plays pit 10
P1 | [6, 6, 0, 0, 5, 5]
   | 2                 1
P0 | [5, 5, 4, 4, 0, 5]
----------------------------------------
Turn 3 | Player 0
AlphaCapture plays pit 5
P1 | [6, 6, 1, 1, 6, 6]
   | 2                 2
P0 | [5, 5, 4, 4, 0, 0]
----------------------------------------
Turn 4 | Player 1
CaptureZero plays pit 12
P1 | [0, 6, 1, 1, 6, 6]
   | 3                 2
P0 | [6, 6, 5, 5, 1, 0]
----------------------------------------
Turn 5 | Player 0
AlphaCapture plays pit 4
P1 | [0, 6, 1, 1, 6, 0]
   | 3        

In [82]:
# play the second match (CaptureZero goes first)
play_match(
    start_player=1,
    alpha_depth=1,
    capturezero_model="/content/AlphaCapture/dqn_mancala.pt"
)



=== GAME START ===

P1 | [4, 4, 4, 4, 4, 4]
   | 0                 0
P0 | [4, 4, 4, 4, 4, 4]
----------------------------------------
Turn 0 | Player 1
CaptureZero plays pit 9
P1 | [5, 5, 5, 0, 4, 4]
   | 1                 0
P0 | [4, 4, 4, 4, 4, 4]
----------------------------------------
Turn 1 | Player 1
CaptureZero plays pit 12
P1 | [0, 5, 5, 0, 4, 4]
   | 2                 0
P0 | [5, 5, 5, 5, 4, 4]
----------------------------------------
Turn 2 | Player 0
AlphaCapture plays pit 5
P1 | [0, 5, 5, 1, 5, 5]
   | 2                 1
P0 | [5, 5, 5, 5, 4, 0]
----------------------------------------
Turn 3 | Player 1
CaptureZero plays pit 7
P1 | [0, 6, 6, 2, 6, 0]
   | 8                 1
P0 | [0, 5, 5, 5, 4, 0]
----------------------------------------
Turn 4 | Player 0
AlphaCapture plays pit 1
P1 | [0, 6, 6, 2, 6, 0]
   | 8                 2
P0 | [0, 0, 6, 6, 5, 1]
----------------------------------------
Turn 5 | Player 0
AlphaCapture plays pit 2
P1 | [0, 6, 6, 2, 7, 1]
   | 8         